<a href="https://colab.research.google.com/github/SuMyatMon17/AI_Solution_Development_Project/blob/main/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup and load data

This notebook performs exploratory data analysis (EDA) on the gas monitoring dataset, focusing on data quality, distribution, and feature relationships to guide preprocessing and model development.

In [ ]:
#import all the required libraries
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [6]:
# Download the SQLite database file from a GitHub repository
!wget https://github.com/SuMyatMon17/AI_Solution_Development_Project/blob/main/data/gas_monitoring.db

# Define the path to the downloaded database file
db_path = 'gas_monitoring.db'

# Establish a connection to the SQLite database
conn = sqlite3.connect(db_path)

# Try to read the table as before (assuming 'gas_monitoring' is the table name)
df_raw = pd.read_sql('SELECT * FROM gas_monitoring', conn)

# Close the database connection to release resources
conn.close()

# Display the first 20 rows of the DataFrame to inspect the data
df_raw.head(20)

--2026-06-01 08:10:06--  https://raw.githubusercontent.com/SuMyatMon17/AI_Solution_Development_Project/main/data/gas_monitoring.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1732608 (1.7M) [application/octet-stream]
Saving to: ‘gas_monitoring.db’

gas_monitoring.db   100%[===================>]   1.65M  --.-KB/s    in 0.04s   

2026-06-01 08:10:07 (41.8 MB/s) - ‘gas_monitoring.db’ saved [1732608/1732608]



# Data Overview

In [ ]:
# Display a concise summary of the DataFrame, including data types and non-null values
df_raw.info()

Categorical Data Standardization

In [ ]:
# Standardize 'Time of Day' column
df_raw['Time of Day'] = df_raw['Time of Day'].str.lower().str.strip()

# Standardize 'HVAC Operation Mode' column
df_raw['HVAC Operation Mode'] = df_raw['HVAC Operation Mode'].str.lower().str.strip()

# Standardize 'Ambient Light Level' column
df_raw['Ambient Light Level'] = df_raw['Ambient Light Level'].str.lower().str.strip()

# Standardize 'Activity Level' column, replacing underscores and correcting specific labels
df_raw['Activity Level'] = df_raw['Activity Level'].str.lower().str.strip().str.replace('_', ' ', regex=False).replace({'lowactivity': 'low activity', 'moderateactivity': 'moderate activity'})

Categorical values were standardized by converting text to lowercase, removing extra spaces, and fixing inconsistent labels to improve consistency for EDA and model training.

Rearranging Rows by Session ID

In [ ]:
# Sort the DataFrame by 'Session ID' in ascending order
df_raw = df_raw.sort_values(
    by="Session ID",
    ascending=True
)

# Display the first 20 rows of the sorted DataFrame
df_raw.head(20)

# Missing Data Exploration

In [ ]:
# Check for non-null values in the DataFrame
df_raw.notnull()

In [ ]:
# Calculate the sum of null values for each column
df_raw.isnull().sum()

Key Observation: The missing value analysis shows that several features contain missing data. Humidity has the highest number of missing values (1,928), followed by MetalOxideSensor_Unit2 (1,410), Ambient Light Level (1,054), and CO_GasSensor (834). The remaining features do not contain any missing values. Since these variables are important for understanding environmental conditions and predicting activity levels, appropriate missing value handling will be performed during preprocessing to ensure data quality and improve model performance.

# Duplicate Data Exploration

In [ ]:
# Identify all duplicate rows in the DataFrame
full_duplicates = df_raw.duplicated()

# Print the total number of full row duplicates found
print(
    "Full row duplicates:",
    full_duplicates.sum()
)

# Display the first 20 rows that are identified as full duplicates
df_raw[full_duplicates].head(20)

In [ ]:
# rows involved in exact duplicate groups
dup_mask = df_raw.duplicated(
    subset=list(df_raw.columns),
    keep=False
)

# Create a DataFrame of all exact duplicate rows
all_exact_duplicates = df_raw[dup_mask].copy()

# sort so identical rows appear next to each other
all_exact_duplicates = all_exact_duplicates.sort_values(
    by=list(df_raw.columns)
)

# Print the total number of rows involved in exact duplicate groups
print("Rows involved in exact duplicate groups:", len(all_exact_duplicates))

# Display the duplicate rows
all_exact_duplicates

Key Observation:

The initial exploration identified 525 rows involved in exact duplicate groups within the raw dataset. These duplicate records may represent either repeated environmental measurements under stable indoor conditions or duplicated entries introduced during data collection and storage.

Since this analysis was performed on the raw dataset, further preprocessing and investigation will be required before deciding whether the duplicate records should be retained or removed for model training.

# Outliers Exploration

In [ ]:
# Display descriptive statistics for numerical columns
df_raw.describe()

In [ ]:
# Selects all numerical columns from the DataFrame
numerical_cols = df_raw.select_dtypes(include=np.number).columns

# Initialize a dictionary to store outlier counts for each column
outliers_count = {}
# Iterate through each numerical column
for col in numerical_cols:
    # Calculate the first quartile (Q1)
    Q1 = df_raw[col].quantile(0.25)
    # Calculate the third quartile (Q3)
    Q3 = df_raw[col].quantile(0.75)
    # Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1
    # Define the lower bound for outlier detection
    lower_bound = Q1 - 1.5 * IQR
    # Define the upper bound for outlier detection
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers for the current column using the IQR method
    col_outliers = df_raw[(df_raw[col] < lower_bound) | (df_raw[col] > upper_bound)]
    # Store the count of outliers for the current column
    outliers_count[col] = len(col_outliers)

# Print the total number of outliers found for each numerical column
print("Number of outliers per numerical column (using IQR method):\n", outliers_count)

In [ ]:
# Identify numerical columns that have outliers
numerical_cols_with_outliers = [col for col, count in outliers_count.items() if count > 0]

# Check if any outliers were found
if not numerical_cols_with_outliers:
    print("No numerical columns with outliers were identified based on the previous calculation.")
else:
    num_plots = len(numerical_cols_with_outliers)
    num_cols = 3 # Number of columns for subplots
    num_rows = (num_plots + num_cols - 1) // num_cols # Calculate number of rows needed

    # Create a figure and a set of subplots
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows))
    axes = axes.flatten() # Flatten the array of axes for easy iteration

    # Iterate through each column with outliers and create a box plot
    for i, col in enumerate(numerical_cols_with_outliers):
        sns.boxplot(y=df_raw[col], ax=axes[i])
        axes[i].set_title(f'Box Plot of {col}')
        axes[i].set_ylabel('') # Remove y-label to avoid clutter, title is sufficient

    # Hide any unused subplots to clean up the display
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # Adjust layout to prevent labels from overlapping
    plt.tight_layout()
    # Display the plots
    plt.show()

Key Observation: The box plots show that several numerical features contain outliers, particularly Temperature, Humidity, CO2_InfraredSensor, CO2_ElectroChemicalSensor, and the Metal Oxide sensors. Some of these values are located far outside the normal data range, which may represent unusual environmental conditions, sensor noise, or contaminated records. The presence of these outliers could affect model performance by influencing feature distributions and prediction results. Therefore, outlier treatment was included in the preprocessing stage using the IQR method to reduce the impact of extreme values while retaining the majority of the data.

# Categorical Data Overview

In [ ]:
# Selects all categorical columns (object type) from the DataFrame
categorical_cols = df_raw.select_dtypes(include='object').columns

# Check if there are any categorical columns
if not categorical_cols.empty:
    # Iterate through each categorical column
    for col in categorical_cols:
        # Print a header for the current column
        print(f"\n--- Categorical Column: {col} ---")
        # Display the value counts for the current column
        print(df_raw[col].value_counts())
        # Print the number of unique values in the current column
        print(f"Number of unique values: {df_raw[col].nunique()}")
else:
    # If no categorical columns are found, print a message
    print("No categorical columns found in the DataFrame.")

In [ ]:
# Bar plot for Time of Day counts
plt.figure(figsize=(8, 6))
sns.countplot(x="Time of Day", data=df_raw, palette="viridis")

plt.title("Count of Records by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

Key Observation:

These initial records are quite evenly distributed across different times of the day, with afternoon having slightly more records while morning, evening, and night have slightly lower but similar counts. This shows that the dataset contains data collected throughout the whole day and is not heavily focused on only one time period reducing the risk of severe time-based sampling bias during Activity Level prediction.

Since this analysis was conducted before handling missing values, duplicates, and outliers, the distribution may change slightly after preprocessing

In [ ]:
# Bar plot for HVAC Operation Mode
plt.figure(figsize=(8, 6))
sns.countplot(x="HVAC Operation Mode", data=df_raw, palette="inferno")

plt.title("Count of Records by HVAC Operation Mode")
plt.xlabel("HVAC Operation Mode")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

Key Observation:

The records are fairly distributed across the different HVAC operation modes with cooling_active and maintenance_mode having slightly higher counts while ventilation_only has the lowest. This suggests that the dataset includes a variety of indoor environmental operating conditions rather than focusing heavily on only one HVAC mode. This balanced distribution may help the model learn activity patterns under different indoor environmental conditions more effectively during training and prediction.

However, since this analysis was conducted before handling missing values, duplicates, and outliers, the distribution may change slightly after preprocessing

In [ ]:
# Bar plot for Ambient Light Level counts
plt.figure(figsize=(8, 6))
sns.countplot(x="Ambient Light Level", data=df_raw, palette="pastel")

plt.title("Count of Records by Ambient Light Level")
plt.xlabel("Ambient Light Level")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

Key Observation:

The initial distribution of records across ambient light levels appears uneven, with very_bright and bright having the highest number of records, while dim and very_dim contain fewer samples. This suggests that the raw dataset currently contains more observations collected under brighter lighting conditions compared to darker environments.  The imbalance may later affect model training, as the model could learn activity patterns more effectively under brighter conditions than dim lighting environments.

As this is still raw data before preprocessing, the distribution may change slightly after data cleaning.

In [ ]:
# Bar plot for Activity Level
plt.figure(figsize=(8, 6))
sns.countplot(x="Activity Level", data=df_raw, palette="plasma")

plt.title("Count of Records by Activity Level")
plt.xlabel("Activity Level")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

Key Observation:

The initial distribution of records across Activity Levels appears imbalanced with low activity having the highest number of records, followed by moderate activity while high activity contains significantly fewer samples. This suggests that the raw dataset currently contains more observations for lower activity situations compared to higher activity conditions. The imbalance may later affect model training, as the model could become more biased toward predicting the dominant low activity category more frequently than minority classes.

These observations are based on the raw dataset and may change slightly after preprocessing.

 Categorizing Numerical Sensor Data


In [ ]:
# Define the bins and labels for categorization
bins = [-float('inf'), 0, 100, float('inf')]
labels = ['below 0', '0-100', 'above 100']

# Categorize Temperature
df_raw['Temperature_Range'] = pd.cut(df_raw['Temperature'], bins=bins, labels=labels, right=False)

# Categorize Humidity
df_raw['Humidity_Range'] = pd.cut(df_raw['Humidity'], bins=bins, labels=labels, right=False)

# Categorize CO2_InfraredSensor
df_raw['CO2_InfraredSensor_Range'] = pd.cut(df_raw['CO2_InfraredSensor'], bins=bins, labels=labels, right=False)

print("--- Temperature Range Counts ---")
print(df_raw['Temperature_Range'].value_counts().sort_index())

print("\n--- Humidity Range Counts ---")
print(df_raw['Humidity_Range'].value_counts().sort_index())

print("\n--- CO2_InfraredSensor Range Counts ---")
print(df_raw['CO2_InfraredSensor_Range'].value_counts().sort_index())

After cleaning in .py, data correlation and visualisation

# Correlation Heat Map

In [ ]:
# Select numerical columns and drop 'Session ID'
num_cols = df_raw.select_dtypes(
    include=np.number
).drop(
    columns=["Session ID"]
)

# Create a new figure for the heatmap
plt.figure(figsize=(12,8))

# Generate a heatmap of the correlation matrix
sns.heatmap(
    num_cols.corr(),
    annot=True,
    cmap="coolwarm"
)

# Set the title of the heatmap
plt.title(
    "Sensor Correlation Heatmap"
)

# Display the plot
plt.show()

Key Observation:

Correlation analysis revealed strong positive correlation among Metal Oxide sensors, particularly between MetalOxideSensor_Unit2 and MetalOxideSensor_Unit4 (r = 0.95). These sensors may capture highly similar environmental information indicating possible feature redundancy which should be investigated later during feature importance analysis.

CO_GasSensor showed strong negative correlations with several Metal Oxide sensors showing that different sensors may respond differently to indoor environmental conditions and potentially provide complementary information for Activity Level prediction.

Temperature and Humidity show weak correlations with most sensor variables that they may contribute more independent environmental information while not overlapping with gas sensor readings. These findings are important for later feature engineering and model development, as highly correlated features may need further investigation through feature importance analysis to determine whether all features are necessary for accurate Activity Level prediction.

However, since this analysis was conducted before handling missing values, duplicates, and outliers, the distribution may change slightly after preprocessing

# Cosine Similarity to check 4 metal oxide sensors for feature engineering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the list of metal oxide sensor columns
metal_cols = [
    "MetalOxideSensor_Unit1",
    "MetalOxideSensor_Unit2",
    "MetalOxideSensor_Unit3",
    "MetalOxideSensor_Unit4"
]

# Create a copy of the relevant columns from the raw DataFrame
metal_df = df_raw[metal_cols].copy()

# Drop rows with missing values for accurate similarity calculation
metal_df = metal_df.dropna()

# Standardize the data to ensure fair comparison
scaler = StandardScaler()
metal_scaled = scaler.fit_transform(metal_df)

# Calculate cosine similarity between the scaled sensor readings
similarity_matrix = cosine_similarity(metal_scaled.T)

# Convert the similarity matrix into a DataFrame for better readability
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=metal_cols,
    columns=metal_cols
)

# Create a new figure for the heatmap visualization
plt.figure(figsize=(8,6))

# Generate a heatmap to visualize the cosine similarity matrix
sns.heatmap(
    similarity_df,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

# Set the title of the heatmap
plt.title("Cosine Similarity Between Metal Oxide Sensors")

# Display the plot
plt.show()

# Print the cosine similarity DataFrame
print(similarity_df)

Key Observation: MetalOxideSensor_Unit2 and MetalOxideSensor_Unit4 show the highest similarity (0.95), meaning they behave very similarly and may contain overlapping information. MetalOxideSensor_Unit3 has the lowest similarity with the other sensors, especially with Unit1 (0.46), suggesting that it behaves differently from the rest of the Metal Oxide sensor group. Based on this observation, PCA was considered during feature engineering to combine the Metal Oxide sensors into a single feature and reduce redundancy. For this analysis, rows with missing Metal Oxide sensor values were removed before calculating cosine similarity.

# Summary

Key findings: (1) Missing values concentrated in Humidity and MetalOxide sensors, (2) 525 duplicate rows, (3) Outliers in several environmental sensors, (4) Imbalanced activity levels. These insights inform preprocessing steps such as imputation, duplicate removal, outlier clipping, PCA for MetalOxide sensors, and class balancing.

In [8]:
!jupyter nbconvert --clear-output --inplace eda.ipynb

[NbConvertApp] WARNING | pattern 'eda.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute